In [2]:
import polars as pl

from postgres_utils import (
    conectar,
    cargar_datos,
    crear_tabla_desde_polars,
    insertar_dataframe,
    insertar_venta,
    ejecutar_consulta
)

In [3]:
conn = conectar()

In [4]:
clientes, productos, ventas = cargar_datos()

print("Clientes:", clientes.shape)
print("Productos:", productos.shape)
print("Ventas:", ventas.shape)

Clientes: (500, 11)
Productos: (150, 6)
Ventas: (10000, 7)


In [5]:
crear_tabla_desde_polars(
    conn,
    clientes,
    "clientes",
    "id_cliente"
)

crear_tabla_desde_polars(
    conn,
    productos,
    "productos",
    "id_producto"
)

crear_tabla_desde_polars(
    conn,
    ventas,
    "ventas",
    "id_venta"
)

In [6]:
tablas = {
    "clientes": clientes,
    "productos": productos,
    "ventas": ventas
}

for nombre_tabla, df in tablas.items():

    resultado = ejecutar_consulta(
        conn,
        f'SELECT COUNT(*) FROM "{nombre_tabla}"'
    )

    cantidad = resultado[0][0]

    if cantidad == 0:

        insertar_dataframe(
            conn,
            df,
            nombre_tabla
        )

        print(
            f"{nombre_tabla}: "
            f"{df.height} registros cargados"
        )

    else:

        print(
            f"{nombre_tabla}: "
            f"{cantidad} registros ya existentes"
        )

clientes: 500 registros cargados
productos: 150 registros cargados
ventas: 10000 registros cargados


In [7]:
for tabla in ["clientes", "productos", "ventas"]:

    resultado = ejecutar_consulta(
        conn,
        f'SELECT COUNT(*) FROM "{tabla}"'
    )

    print(
        f"{tabla}: {resultado[0][0]} registros"
    )

clientes: 500 registros
productos: 150 registros
ventas: 10000 registros


In [8]:
# Última comprobación: ver las últimas ventas

resultado = ejecutar_consulta(
    conn,
    """
    SELECT *
    FROM ventas
    ORDER BY id_venta DESC
    LIMIT 10
    """
)

resultado

[(10000, '2025-02-11 02:00:00', 112, 43, 1, 'Tarjeta', 0),
 (9999, '2025-09-01 08:00:00', 484, 61, 1, 'Tarjeta', 5),
 (9998, '2025-06-14 09:00:00', 43, 126, 5, 'Tarjeta', 10),
 (9997, '2025-08-22 22:00:00', 295, 48, 3, 'Cheque regalo', 10),
 (9996, '2025-06-03 13:00:00', 301, 90, 3, 'Tarjeta', 0),
 (9995, '2025-09-07 16:00:00', 129, 45, 2, 'Tarjeta', 5),
 (9994, '2025-04-12 14:00:00', 72, 138, 4, 'Tarjeta', 5),
 (9993, '2025-06-04 19:00:00', 110, 146, 2, 'Bizum', 0),
 (9992, '2025-01-25 12:00:00', 136, 114, 2, 'Tarjeta', 0),
 (9991, '2025-04-26 07:00:00', 176, 98, 4, 'Tarjeta', 5)]

In [9]:
# Ejecutamos el producer y consumer y probamos 

resultado = ejecutar_consulta(
    conn,
    """
    SELECT *
    FROM ventas
    ORDER BY id_venta DESC
    LIMIT 10
    """
)

resultado

[(10000, '2025-02-11 02:00:00', 112, 43, 1, 'Tarjeta', 0),
 (9999, '2025-09-01 08:00:00', 484, 61, 1, 'Tarjeta', 5),
 (9998, '2025-06-14 09:00:00', 43, 126, 5, 'Tarjeta', 10),
 (9997, '2025-08-22 22:00:00', 295, 48, 3, 'Cheque regalo', 10),
 (9996, '2025-06-03 13:00:00', 301, 90, 3, 'Tarjeta', 0),
 (9995, '2025-09-07 16:00:00', 129, 45, 2, 'Tarjeta', 5),
 (9994, '2025-04-12 14:00:00', 72, 138, 4, 'Tarjeta', 5),
 (9993, '2025-06-04 19:00:00', 110, 146, 2, 'Bizum', 0),
 (9992, '2025-01-25 12:00:00', 136, 114, 2, 'Tarjeta', 0),
 (9991, '2025-04-26 07:00:00', 176, 98, 4, 'Tarjeta', 5)]

## Pipeline con Prefect

El flujo `pipeline_datos` consume las ventas de Kafka, las transforma con Polars y las guarda en PostgreSQL.

In [ ]:
from pipeline_datos import pipeline_datos

pipeline_datos(segundos=20)

### Programar el pipeline

Para ejecutarlo automáticamente cada 60 segundos:

```bash
python servir_pipeline.py
```